# Practice 1 (Part 1): Fashion-MNIST Data Pipeline, Tensors & Transforms

## PyTorch Tensors, Torchvision Transforms, Dataset Ingestion & Exploratory Data Analysis
This notebook establishes the foundational data layer for Fashion-MNIST classification in PyTorch. It fulfills the first core requirement of Practice 1 by mastering multi-dimensional tensors, dataset ingestion, Z-score normalization transforms, data augmentation, batch generation with DataLoader, and pixel geometry exploration.

## Roadmap Table
| Step | Description | What it does | Import path |
|---|---|---|---|
| 1 | Environment & Seed Setup | Configures dynamic device and seed for reproducibility | [`PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md) |
| 2 | PyTorch Tensors & Mechanics | Demonstrates tensor manipulation, shapes, and GPU transfer | `torch` |
| 3 | Dataset Ingestion | Loads Fashion-MNIST dataset from Zalando Research | `torchvision.datasets.FashionMNIST` |
| 4 | Transforms & Augmentation | Implements ToTensor, Z-score normalization, and random horizontal flip | `torchvision.transforms` |
| 5 | DataLoader & Batching | Configures batch generator with shuffling and pinned memory | `torch.utils.data.DataLoader` |
| 6 | Exploratory Data Analysis | Visualizes class balance, pixel histogram, and 16-sample batch grid | `matplotlib.pyplot` |
---

## References
- Task Brief: [`labs/lab1/QUESTION.md`](../QUESTION.md)
- Pipeline Architecture: [`docs/lab1/PIPELINE_EXPLANATION.md`](../../../docs/lab1/PIPELINE_EXPLANATION.md)
- Presentation Script: [`docs/lab1/PRESENTATION.md`](../../../docs/lab1/PRESENTATION.md)
- Dataset Hub: [Zalando Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist)
- Framework Governance: [`agents/rules/PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md)
- Notebook Standards: [`agents/rules/NOTEBOOK_HEADER_CONVENTION.md`](../../../.agents/rules/NOTEBOOK_HEADER_CONVENTION.md)


## 1. Environment & Reproducibility Setup

Tuân thủ quy chuẩn [`PYTORCH_FRAMEWORK_RULES.md`](../../../.agents/rules/PYTORCH_FRAMEWORK_RULES.md), ta thiết lập cấu hình thiết bị động (`device agnostic`) và khóa ngẫu nhiên (seed reproducibility) trên cả Python, NumPy, và PyTorch để đảm bảo kết quả thực nghiệm hoàn toàn có thể tái lập.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] PyTorch Version : {torch.__version__}")
print(f"[Setup] Active Device   : {device}")
if torch.cuda.is_available():
    print(f"[Setup] GPU Model       : {torch.cuda.get_device_name(0)}")


## 2. Làm Chủ PyTorch Tensors & Thao Tác Cơ Bản

Tensor là cấu trúc dữ liệu cốt lõi của PyTorch, tương tự như mảng NumPy ndarray nhưng được tối ưu hóa cho phép tính nhân ma trận trên GPU và tích hợp sẵn đồ thị đạo hàm tự động (Autograd).

In [ ]:
# 1. Khởi tạo tensor từ dữ liệu Python list và NumPy array
sample_data = [[1.0, 2.0], [3.0, 4.0]]
tensor_a = torch.tensor(sample_data, dtype=torch.float32)

# 2. Các thuộc tính then chốt của Tensor
print("Tensor A:\n", tensor_a)
print(f"Shape    : {tensor_a.shape}")
print(f"Dtype    : {tensor_a.dtype}")
print(f"Device   : {tensor_a.device}")

# 3. Thao tác chuyển đổi thiết bị (Device Transfer)
tensor_gpu = tensor_a.to(device)
print(f"Device sau khi chuyển: {tensor_gpu.device}")

# 4. Thao tác tái định hình (Reshape / View / Squeeze / Unsqueeze)
dummy_img = torch.randn(28, 28) # Ảnh 28x28 2D
img_4d = dummy_img.unsqueeze(0).unsqueeze(0) # Chuyển thành [Batch=1, Channel=1, H=28, W=28]
print("Shape ảnh sau unsqueeze:", img_4d.shape)


## 3. Thu Nạp Dữ Liệu Fashion-MNIST (Data Ingestion)

Bộ dữ liệu **Fashion-MNIST** được phát triển bởi Zalando Research, gồm 70,000 ảnh trang phục đơn sắc kích thước $28 	imes 28$ thuộc 10 danh mục. Dữ liệu được tải về và lưu trữ bất biến (Immutable Raw Zone) trong thư mục `data/`.

In [ ]:
CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

# Nạp bản thô ToTensor để tính toán thống kê EDA
raw_train = FashionMNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
raw_test  = FashionMNIST(root="data", train=False, download=True, transform=transforms.ToTensor())

print(f"Số lượng ảnh Train : {len(raw_train):,}")
print(f"Số lượng ảnh Test  : {len(raw_test):,}")
print(f"Kích thước ảnh mẫu : {raw_train[0][0].shape} (Channels, Height, Width)")
print(f"Nhãn của ảnh mẫu   : {raw_train[0][1]} ({CLASSES[raw_train[0][1]]})")


## 4. Chuỗi Biến Đổi Transforms & Chuẩn Hóa Z-Score

Để mạng nơ-ron hội tụ nhanh và ổn định, ta áp dụng chuẩn hóa Z-Score: $\mathbf{x}' = \frac{\mathbf{x} - \mu}{\sigma}$. Thống kê từ 60,000 ảnh train cho thấy giá trị trung bình là $\mu = 0.2860$ và độ lệch chuẩn $\sigma = 0.3530$.

Nhóm xây dựng hai chuỗi biến đổi riêng biệt:
- `train_transform`: Tích hợp Data Augmentation (`RandomHorizontalFlip`, `RandomCrop`) nhằm tăng cường độ bền vững hình học.
- `test_transform`: Chỉ chuẩn hóa cơ bản để đo lường khách quan.

In [ ]:
MEAN = (0.2860,)
STD  = (0.3530,)

train_transform = transforms.Compose([
    transforms.RandomCrop(28, padding=2),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Nạp lại dataset với chuỗi transforms chính thức
train_dataset = FashionMNIST(root="data", train=True, download=True, transform=train_transform)
test_dataset  = FashionMNIST(root="data", train=False, download=True, transform=test_transform)
print("Đã cấu hình thành công chuỗi Transforms cho Train và Test!")


## 5. Xây Dựng PyTorch `DataLoader` Đa Luồng & Ghim Bộ Nhớ

`DataLoader` đóng gói việc phân lô (batching), xáo trộn ngẫu nhiên (shuffling) và nạp dữ liệu đa tiến trình (`num_workers`). Thiết lập `pin_memory=True` giúp tăng tốc độ truyền tensor từ RAM sang VRAM qua bus PCIe.

In [ ]:
BATCH_SIZE = 64
NUM_WORKERS = 2

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

# Kiểm tra kích thước một batch nạp ra
images_batch, labels_batch = next(iter(train_loader))
print(f"Tensor images_batch shape : {images_batch.shape} (B, C, H, W)")
print(f"Tensor labels_batch shape : {labels_batch.shape} (B)")


## 6. Khảo Sát Hình Học Dữ Liệu Thực Tế (Exploratory Data Analysis - EDA)

Thực hiện trực quan hóa phân bố nhãn lớp, biểu đồ histogram cường độ pixel, và lưới 16 ảnh thực tế đại diện cho 10 danh mục trang phục.

In [ ]:
# 1. Đếm tần suất các lớp
labels_list = [raw_train[i][1] for i in range(len(raw_train))]
counts = np.bincount(labels_list)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(CLASSES, counts, color="#1f77b4", edgecolor="black")
ax[0].set_title("Phân phối 10 Nhãn Lớp (Train Split - 60,000 ảnh)")
ax[0].set_ylabel("Số lượng mẫu")
ax[0].tick_params(axis='x', rotation=45)

# 2. Histogram cường độ pixel của ảnh thô
raw_pixels = raw_train.data.numpy().flatten()
ax[1].hist(raw_pixels, bins=50, color="#ff7f0e", edgecolor="black", range=(0, 255))
ax[1].set_title("Histogram Phân phối Pixel Thô [0, 255]")
ax[1].set_xlabel("Cường độ pixel")
ax[1].set_ylabel("Tần số")
plt.tight_layout()
plt.show()

# 3. Trực quan hóa Lưới 16 ảnh mẫu kèm nhãn lớp
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for idx, ax_i in enumerate(axes.flatten()):
    img, lbl = raw_train[idx]
    ax_i.imshow(img.squeeze().numpy(), cmap="gray")
    ax_i.set_title(f"{CLASSES[lbl]}", fontsize=9)
    ax_i.axis("off")
plt.suptitle("Lưới 16 Ảnh Mẫu Fashion-MNIST (28x28 Grayscale)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
